# Fuzzy Based Automatic Power Factor Controller (APFC)

This Python notebook simulates a fuzzy logic based APFC system. The controller uses **Power Factor (PF)** and **Reactive Power (Q)** as inputs and determines the required capacitor compensation level.

**Inputs:** PF = 0.5 to 1.0, Reactive Power = 0 to 100 kVAr  
**Output:** Capacitor step command = 0 to 4

In [ ]:
# Install required library in Google Colab if needed
!pip -q install scikit-fuzzy


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from skfuzzy import control as ctrl


In [ ]:
# Define fuzzy variables
pf = ctrl.Antecedent(np.arange(0.5, 1.001, 0.001), 'power_factor')
q = ctrl.Antecedent(np.arange(0, 101, 1), 'reactive_power')
cap = ctrl.Consequent(np.arange(0, 4.01, 0.01), 'capacitor_step')

# Membership functions for PF
pf['very_low'] = fuzz.trimf(pf.universe, [0.50, 0.50, 0.65])
pf['low']      = fuzz.trimf(pf.universe, [0.60, 0.70, 0.80])
pf['medium']   = fuzz.trimf(pf.universe, [0.75, 0.82, 0.90])
pf['high']     = fuzz.trimf(pf.universe, [0.87, 0.93, 0.98])
pf['very_high']= fuzz.trimf(pf.universe, [0.95, 1.00, 1.00])

# Membership functions for reactive power
q['low']    = fuzz.trimf(q.universe, [0, 0, 30])
q['medium'] = fuzz.trimf(q.universe, [20, 45, 70])
q['high']   = fuzz.trimf(q.universe, [60, 80, 100])

# Output membership functions: capacitor steps 0 to 4
cap['off'] = fuzz.trimf(cap.universe, [0, 0, 1])
cap['step1'] = fuzz.trimf(cap.universe, [0.5, 1, 1.5])
cap['step2'] = fuzz.trimf(cap.universe, [1.5, 2, 2.5])
cap['step3'] = fuzz.trimf(cap.universe, [2.5, 3, 3.5])
cap['step4'] = fuzz.trimf(cap.universe, [3.5, 4, 4])


In [ ]:
# Display membership functions
pf.view()
plt.show()

q.view()
plt.show()

cap.view()
plt.show()


## 25 Fuzzy Rules

Five PF levels × five effective compensation levels are represented using a 25-rule rule base. Reactive-power membership is also used to strengthen the compensation decision.

In [ ]:
# 25-rule fuzzy rule base
# Each row corresponds to PF level; columns correspond to compensation demand levels.
pf_levels = ['very_low', 'low', 'medium', 'high', 'very_high']
q_levels = ['low', 'medium', 'high']

# 5 x 5 rule table for a second compensation index derived from Q.
rule_outputs = [
    ['step2', 'step3', 'step4', 'step4', 'step4'],
    ['step1', 'step2', 'step3', 'step4', 'step4'],
    ['off',   'step1', 'step2', 'step3', 'step4'],
    ['off',   'off',   'step1', 'step2', 'step3'],
    ['off',   'off',   'off',   'step1', 'step2']
]

# Create 25 rules using PF and a five-level reactive-power demand index.
# The Q membership functions are combined into five demand levels by scaling Q.
q_demand = ctrl.Antecedent(np.arange(0, 101, 1), 'q_demand')
q_demand['q1'] = fuzz.trimf(q_demand.universe, [0, 0, 25])
q_demand['q2'] = fuzz.trimf(q_demand.universe, [10, 25, 40])
q_demand['q3'] = fuzz.trimf(q_demand.universe, [30, 45, 60])
q_demand['q4'] = fuzz.trimf(q_demand.universe, [50, 65, 80])
q_demand['q5'] = fuzz.trimf(q_demand.universe, [70, 100, 100])

rules = []
q_names = ['q1', 'q2', 'q3', 'q4', 'q5']
for i, pf_name in enumerate(pf_levels):
    for j, q_name in enumerate(q_names):
        rules.append(ctrl.Rule(pf[pf_name] & q_demand[q_name], cap[rule_outputs[i][j]]))

print('Number of fuzzy rules:', len(rules))


In [ ]:
# Build and simulate the fuzzy controller
apfc_system = ctrl.ControlSystem(rules)
apfc = ctrl.ControlSystemSimulation(apfc_system)

def apfc_controller(power_factor, reactive_power):
    """Return the fuzzy capacitor-step command for PF and reactive power."""
    power_factor = float(np.clip(power_factor, 0.5, 1.0))
    reactive_power = float(np.clip(reactive_power, 0, 100))
    sim = ctrl.ControlSystemSimulation(apfc_system)
    sim.input['power_factor'] = power_factor
    sim.input['reactive_power'] = reactive_power
    # q_demand is the same physical reactive-power signal used for the 5-level rule base.
    sim.input['q_demand'] = reactive_power
    sim.compute()
    return float(sim.output['capacitor_step'])

# Test cases
test_cases = [
    (0.55, 90), (0.60, 70), (0.65, 50), (0.70, 30), (0.75, 20),
    (0.80, 80), (0.85, 60), (0.88, 40), (0.90, 20), (0.92, 70),
    (0.94, 50), (0.96, 30), (0.97, 10), (0.98, 80), (1.00, 5)
]

print('PF\tQ (kVAr)\tCapacitor command')
for pf_value, q_value in test_cases:
    print(f'{pf_value:.2f}\t{q_value:>7.1f}\t{apfc_controller(pf_value, q_value):.2f}')


In [ ]:
# Plot controller response for different reactive-power values
pf_values = np.linspace(0.5, 1.0, 60)
q_values = [20, 50, 80]

plt.figure(figsize=(9, 5))
for q_value in q_values:
    outputs = [apfc_controller(p, q_value) for p in pf_values]
    plt.plot(pf_values, outputs, label=f'Q = {q_value} kVAr')

plt.xlabel('Power Factor')
plt.ylabel('Fuzzy Capacitor Step Command')
plt.title('Fuzzy APFC Controller Response')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# Simple 10-case analysis table
cases = [
    (0.55, 90), (0.60, 75), (0.65, 60), (0.70, 45), (0.75, 30),
    (0.80, 70), (0.85, 55), (0.90, 40), (0.95, 20), (0.99, 5)
]

print(f'{"Case":<6}{"PF":<10}{"Q(kVAr)":<12}{"Capacitor Step":<18}')
print('-' * 46)
for i, (p, qv) in enumerate(cases, 1):
    result = apfc_controller(p, qv)
    step = int(np.clip(np.rint(result), 0, 4))
    print(f'{i:<6}{p:<10.2f}{qv:<12.1f}{step:<18}')


## Conclusion

The fuzzy APFC controller uses power factor and reactive power information to select one of four capacitor compensation steps. The simulation demonstrates how fuzzy logic can provide a smooth and rule-based decision for automatic power-factor correction.